<a href="https://colab.research.google.com/github/vakharevedant-cmd/purchase-pattern-analysis/blob/main/Apriori2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Market Basket Analysis**

**Importing** **Libraries**


In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import networkx as nx
import matplotlib.pyplot as plt
from pandas.plotting import parallel_coordinates
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import KMeans
import plotly.graph_objects as go

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Market_basket_analysis.xlsx to Market_basket_analysis.xlsx


**Data** **Exploration** **and** **Preparation**

In [ ]:
purchase_df  = pd.read_excel('Market_basket_analysis.xlsx', dtype={'BillNo': str})
purchase_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31432 entries, 0 to 31431
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   BillNo        31432 non-null  object        
 1   Itemname      31432 non-null  object        
 2   Quantity      31432 non-null  int64         
 3   Present_Date  31432 non-null  datetime64[ns]
 4   Price         31432 non-null  float64       
 5   CustomerID    31432 non-null  int64         
 6   Country       31432 non-null  object        
 7   Revenue       31432 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(3)
memory usage: 1.9+ MB


In [ ]:
purchase_df['CustomerID'] = purchase_df['CustomerID'].astype('str')
purchase_df['BillNo'] = purchase_df['BillNo'].astype('object')
purchase_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31432 entries, 0 to 31431
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   BillNo        31432 non-null  object        
 1   Itemname      31432 non-null  object        
 2   Quantity      31432 non-null  int64         
 3   Present_Date  31432 non-null  datetime64[ns]
 4   Price         31432 non-null  float64       
 5   CustomerID    31432 non-null  object        
 6   Country       31432 non-null  object        
 7   Revenue       31432 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 1.9+ MB


In [ ]:
# Keep only numeric columns for outlier removal
df_numeric = purchase_df.select_dtypes(include='number')

Q1 = df_numeric.quantile(0.25)
Q3 = df_numeric.quantile(0.75)
IQR = Q3 - Q1

# Filter out rows that contain outliers in any numeric column
mask = ~((df_numeric < (Q1 - 1.5 * IQR)) | (df_numeric > (Q3 + 1.5 * IQR))).any(axis=1)
df_filtered = purchase_df[mask]

In [ ]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25595 entries, 0 to 31431
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   BillNo        25595 non-null  object        
 1   Itemname      25595 non-null  object        
 2   Quantity      25595 non-null  int64         
 3   Present_Date  25595 non-null  datetime64[ns]
 4   Price         25595 non-null  float64       
 5   CustomerID    25595 non-null  object        
 6   Country       25595 non-null  object        
 7   Revenue       25595 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 1.8+ MB


In [ ]:
df_filtered.nunique()

,0
BillNo,1426
Itemname,2212
Quantity,24
Present_Date,1341
Price,67
CustomerID,912
Country,17
Revenue,348


In [ ]:
# Convert the column to datetime with the specified format
df_filtered.loc[:, 'Present_Date'] = pd.to_datetime(df_filtered['Present_Date'], format='%d.%m.%Y %H:%M')

In [ ]:
df_filtered.head()

,BillNo,Itemname,Quantity,Present_Date,Price,CustomerID,Country,Revenue
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [ ]:
df_filtered[['Quantity', 'Price','Revenue']].describe().round(2)

,Quantity,Price,Revenue
count,25591.00,25591.00,25591.00
mean,6.64,2.27,10.98
std,6.53,1.50,9.02
min,1.00,0.12,0.14
25%,2.00,1.25,3.75
50%,4.00,1.95,8.70
75%,12.00,2.95,15.60
max,27.00,7.50,42.50


In [ ]:
df_filtered=df_filtered.loc[purchase_df['Quantity']>0]
df_filtered=df_filtered.loc[purchase_df['Price']>0]

In [ ]:
transaction = df_filtered.groupby(['BillNo', 'Present_Date'])['Itemname'].apply(lambda x: ','.join(x)).reset_index()
transaction.head()

,BillNo,Present_Date,Itemname
0,536365,2010-12-01 08:26:00,"WHITE HANGING HEART T-LIGHT HOLDER,WHITE METAL..."
1,536366,2010-12-01 08:28:00,"HAND WARMER UNION JACK,HAND WARMER RED POLKA DOT"
2,536367,2010-12-01 08:34:00,"POPPY'S PLAYHOUSE BEDROOM,POPPY'S PLAYHOUSE KI..."
3,536368,2010-12-01 08:34:00,"JAM MAKING SET WITH JARS,RED COAT RACK PARIS F..."
4,536369,2010-12-01 08:35:00,BATH BUILDING BLOCK WORD


In [ ]:
transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1428 entries, 0 to 1427
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   BillNo        1428 non-null   object        
 1   Present_Date  1428 non-null   datetime64[ns]
 2   Itemname      1428 non-null   object        
dtypes: datetime64[ns](1), object(2)
memory usage: 33.6+ KB


In [ ]:
# Drop unnecessary columns
transaction.drop(columns=['BillNo', 'Present_Date'], inplace=True)

In [ ]:
# Split the 'Itemname' column into separate rows
transaction = transaction.assign(Itemname=transaction['Itemname'].str.split(',')).explode('Itemname')

# Create a crosstab (binary matrix) of items per transaction
basket_encoded = pd.crosstab(index=transaction.index, columns=transaction['Itemname'])

# Group by the transaction index and convert the counts of items done by pd.crosstab() into a simple presence or absence
basket_encoded = basket_encoded.groupby(level=0).apply(lambda x: x > 0)

In [ ]:
basket_encoded

,Itemname,,1 HANGER,BACK DOOR,BAROQUE,BILLBOARD FONTS DESIGN,BIRTHDAY CARD,BLUE,CAROUSEL,CHOCOLATE SPOTS,CHOCOLATE SPOTS,...,YOU'RE CONFUSING ME METAL SIGN,YULETIDE IMAGES GIFT WRAP SET,YULETIDE IMAGES S/6 PAPER BOXES,ZINC FINISH 15CM PLANTER POTS,ZINC HEART LATTICE CHARGER LARGE,ZINC HEART LATTICE CHARGER SMALL,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC WILLIE WINKIE CANDLE STICK,pack/12
row_0,row_0,,,,,,,,,,,,,,,,,,,,,
0,0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1423,1423,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1424,1424,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1425,1425,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# **Association** **Rule** **Mining**

In [ ]:
# Association Rule Mining
frequent_itemsets = apriori(basket_encoded, min_support=0.01, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

In [ ]:
# Display information of the rules
print("Association Rules:")
rules.head()

Association Rules:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(),(FANCY FONT BIRTHDAY CARD),0.025910,0.018908,0.018908,0.729730,38.594595,1.0,0.018418,3.630042,1.000000,0.729730,0.724521,0.864865
1,(FANCY FONT BIRTHDAY CARD),(),0.018908,0.025910,0.018908,1.000000,38.594595,1.0,0.018418,inf,0.992862,0.729730,1.000000,0.864865
2,( CHOCOLATE SPOTS),(SWISS ROLL TOWEL),0.011204,0.014006,0.011204,1.000000,71.400000,1.0,0.011048,inf,0.997167,0.800000,1.000000,0.900000
3,(SWISS ROLL TOWEL),( CHOCOLATE SPOTS),0.014006,0.011204,0.011204,0.800000,71.400000,1.0,0.011048,4.943978,1.000000,0.800000,0.797734,0.900000
4,(KEY FOB ),( SHED),0.013305,0.010504,0.010504,0.789474,75.157895,1.0,0.010364,4.700105,1.000000,0.789474,0.787239,0.894737


# **Market Basket Analysis - Support vs. Confidence**

In [ ]:
# Convert frozensets to lists for serialization
rules['antecedents'] = rules['antecedents'].apply(list)
rules['consequents'] = rules['consequents'].apply(list)

# Create a scatter plot using plotly.graph_objects
fig = go.Figure()

# Add scatter trace
fig.add_trace(go.Scatter(
    x=rules['support'],
    y=rules['confidence'],
    mode='markers',
    marker=dict(
        size=rules['lift'] * 1,  # Scale size by lift for better visibility
        color=rules['lift'],
        colorscale='Cividis',  # Use 'Cividis' for a visually appealing and colorblind-friendly colorscale
        colorbar=dict(
            title=dict(
                text='Lift',  # Add "Lift" label to colorbar legend
                font=dict(color='teal')  # Set "Lift" label to teal
            ),
            tickfont=dict(color='plum'),  # Set tick labels to plum
            ticks='outside',
        ),
        showscale=True
    ),
    text=[
        f'Antecedents: {a}<br>'
        f'Consequents: {c}<br>'
        f'Support: {support:.2f}<br>'
        f'Confidence: {confidence:.2f}<br>'
        f'Lift: {lift:.2f}'
        for a, c, support, confidence, lift in zip(rules['antecedents'], rules['consequents'], rules['support'], rules['confidence'], rules['lift'])
    ],
    showlegend=False,
    hoverinfo='text',
    name='Rules'
))

# Update layout with titles and axis labels
fig.update_layout(
    title={
        'text': 'Market Basket Analysis - Support vs. Confidence',
        'font': {'size': 16, 'color': 'teal'}  # Set title font color to teal
    },
    xaxis_title='Support',
    yaxis_title='Confidence',
    showlegend=True,
    template='plotly_white',  # Use a white template for a clean look
    xaxis=dict(
        title_font=dict(size=14, family='Arial', color='teal'),
        tickfont=dict(size=12, color='plum')
    ),
    yaxis=dict(
        title_font=dict(size=14, family='Arial', color='teal'),
        tickfont=dict(size=12, color='plum')
    ),
    paper_bgcolor='#2A2A2A',  # Set background color to a dark gray, close to black
    plot_bgcolor='#2C2C2C'    # Set plot area background color to light blue
)

# Show the interactive plot
fig.show()

In [ ]:
# Convert frozensets to lists
rules['antecedents'] = rules['antecedents'].apply(list)
rules['consequents'] = rules['consequents'].apply(list)

# Calculate rule length (total number of items in antecedents + consequents)
rules['rule_length'] = rules['antecedents'].apply(len) + rules['consequents'].apply(len)

# Create 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=rules['support'],
    y=rules['confidence'],
    z=rules['lift'],
    mode='markers',
    marker=dict(
        size=rules['rule_length'] * 2,  # Emphasize rule complexity
        color=rules['lift'],
        colorscale='Cividis',
        colorbar=dict(
            title=dict(text='Lift', font=dict(color='teal')),
            tickfont=dict(color='plum'),
            ticks='outside'
        ),
        opacity=0.7
    ),
    text=[
        f'Antecedents: {a}<br>'
        f'Consequents: {c}<br>'
        f'Support: {support:.2f}<br>'
        f'Confidence: {confidence:.2f}<br>'
        f'Lift: {lift:.2f}<br>'
        f'Rule Length: {length}'
        for a, c, support, confidence, lift, length in zip(
            rules['antecedents'],
            rules['consequents'],
            rules['support'],
            rules['confidence'],
            rules['lift'],
            rules['rule_length']
        )
    ],
    hoverinfo='text',
    showlegend=False
)])

# Update layout
fig.update_layout(
    title=dict(
        text='Market Basket Analysis - 3D View',
        font=dict(size=16, color='teal')
    ),
    scene=dict(
        xaxis_title='Support',
        yaxis_title='Confidence',
        zaxis_title='Lift',
        xaxis=dict(title_font=dict(size=14, color='teal'), tickfont=dict(size=12, color='plum')),
        yaxis=dict(title_font=dict(size=14, color='teal'), tickfont=dict(size=12, color='plum')),
        zaxis=dict(title_font=dict(size=14, color='teal'), tickfont=dict(size=12, color='plum'))
    ),
    paper_bgcolor='#2A2A2A',
    plot_bgcolor='#2C2C2C',
    template='plotly_white'
)

# Show plot
fig.show()

# **Interactive Network Graph of Association Rules**


In [ ]:
# Create a directed graph
G = nx.DiGraph()

# Add edges with support, confidence, lift, and rule name as edge attributes
for index, row in rules.iterrows():
    antecedent = str(row['antecedents'])  # Ensure it's a string
    consequent = str(row['consequents'])  # Ensure it's a string
    support = row['support']
    confidence = row['confidence']
    lift = row['lift']
    rule_name = f"{antecedent} -> {consequent}"
    G.add_edge(antecedent, consequent, support=support, confidence=confidence, lift=lift, rule_name=rule_name)

    # Add or update node attributes to ensure they have both antecedent and consequent if applicable
    if not G.nodes[antecedent].get('antecedent'):
        G.nodes[antecedent]['antecedent'] = antecedent
    if not G.nodes[antecedent].get('consequent'):
        G.nodes[antecedent]['consequent'] = consequent
    G.nodes[antecedent]['support'] = support
    G.nodes[antecedent]['confidence'] = confidence
    G.nodes[antecedent]['lift'] = lift

    if not G.nodes[consequent].get('antecedent'):
        G.nodes[consequent]['antecedent'] = antecedent
    if not G.nodes[consequent].get('consequent'):
        G.nodes[consequent]['consequent'] = consequent
    G.nodes[consequent]['support'] = support
    G.nodes[consequent]['confidence'] = confidence
    G.nodes[consequent]['lift'] = lift

# Get position of nodes using spring layout
pos = nx.spring_layout(G, k=0.6, iterations=50)

# Create edge traces
edge_trace = []
for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]

    # Create a trace for each edge
    edge_trace.append(
        go.Scatter(
            x=[x0, x1, None],
            y=[y0, y1, None],
            mode='lines',
            line=dict(width=edge[2]['support']*80, color='#FFFFFF'),  # Edge color set to white
            hoverinfo='text',
            text=f"Support: {edge[2]['support']:.2f}<br>Confidence: {edge[2]['confidence']:.2f}<br>Lift: {edge[2]['lift']:.2f}<br>Rule: {edge[2]['rule_name']}",
            opacity=0.6
        )
    )

# Create node trace with chocolate color and detailed hover text
node_trace = go.Scatter(
    x=[pos[node][0] for node in G.nodes()],
    y=[pos[node][1] for node in G.nodes()],
    mode='markers',
    hoverinfo='text',
    text=[f"Antecedent: {G.nodes[node].get('antecedent', 'N/A')}<br>Consequent: {G.nodes[node].get('consequent', 'N/A')}<br>Support: {G.nodes[node].get('support', 'N/A')}<br>Confidence: {G.nodes[node].get('confidence', 'N/A')}<br>Lift: {G.nodes[node].get('lift', 'N/A')}"
          for node in G.nodes()],
    marker=dict(
        showscale=False,
        color='#40E0D0',  # Node color set to turquoise blue
        size=20,
        opacity=0.6,
        line=dict(width=0)  # No border for nodes
    )
)

# Create the figure
fig = go.Figure(data=edge_trace + [node_trace],
                layout=go.Layout(
                    title='Interactive Network Graph of Association Rules',
                    titlefont=dict(size=16, color='#D2B48C'),  # Title color set to leather
                    showlegend=False,
                    hovermode='closest',
                    margin=dict(b=0, l=0, r=0, t=40),
                    xaxis=dict(
                        showgrid=False,
                        zeroline=False,
                        tickfont=dict(color='#D2B48C'),  # X-axis tick labels color
                        titlefont=dict(color='#D2B48C')  # X-axis title color
                    ),
                    yaxis=dict(
                        showgrid=False,
                        zeroline=False,
                        tickfont=dict(color='#D2B48C'),  # Y-axis tick labels color
                        titlefont=dict(color='#D2B48C')  # Y-axis title color
                    ),
                    paper_bgcolor='#2A2A2A',  # Background color
                    plot_bgcolor='#2A2A2A'    # Plot area color
                ))

# Show the figure
fig.show()


## **RFM Analysis**


In [ ]:
purchase_df_rfm = purchase_df

# Define today’s date for recency calculation
today_date = purchase_df_rfm['Present_Date'].max()

# Group by CustomerID to calculate RFM metrics
rfm = purchase_df_rfm.groupby('CustomerID').agg({
    'Present_Date': lambda x: (today_date - x.max()).days,   # Recency
    'BillNo': 'nunique',                             # Frequency (count unique orders)
    'Revenue': 'sum'                             # Monetary (sum of total price)
})

In [ ]:
# Rename columns for clarity
rfm.columns = ['recency', 'frequency', 'monetary']

# Reset index for further analysis
rfm = rfm.reset_index()

In [ ]:
# Check the results
rfm.head()

,CustomerID,recency,frequency,monetary
0,12347,33,1,711.79
1,12370,24,1,277.20
2,12377,21,1,1001.52
3,12383,18,1,600.72
4,12386,4,2,401.90


## **Cluster Analysis**

In [ ]:
# Suppress specific warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

In [ ]:
# Set number of clusters (k) and fit the model
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
kmeans.fit(rfm_scaled)

# Add the cluster labels to the original RFM DataFrame
rfm['Cluster'] = kmeans.labels_

# Check the distribution of customers in each cluster
rfm['Cluster'].value_counts()

,count
Cluster,
0,739
1,237
2,8


## **Cluster Profile**

In [ ]:
cluster_profile = rfm.groupby('Cluster').agg({
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean',
    'CustomerID': 'count'  # Count of customers in each cluster
}).rename(columns={'CustomerID': 'num_customers'})

In [ ]:
cluster_profile

,recency,frequency,monetary,num_customers
Cluster,,,,
0,30.259811,1.322057,425.386644,739
1,4.105485,2.240506,1155.099241,237
2,16.625000,14.875000,12016.512500,8


## **Recency vs Frequency by Cluster**

In [ ]:
# Define new colors for clusters
colors = {0: '#4169E1',  # Royal Blue
          1: '#DC143C',  # Crimson Red
          2: '#3CB371'}  # Medium Sea Green

# Create a Plotly scatter plot with fixed point size
fig = go.Figure()

for cluster in rfm['Cluster'].unique():
    cluster_data = rfm[rfm['Cluster'] == cluster]
    fig.add_trace(go.Scatter(
        x=cluster_data['recency'],
        y=cluster_data['frequency'],
        mode='markers',
        marker=dict(
            size=12,  # Set fixed size for points
            color=colors[cluster],
            line=dict(width=1, color='white')  # Optional: add a white border to points
        ),
        name=f'Cluster {cluster}'
    ))

# Update layout to match desired style
fig.update_layout(
    paper_bgcolor='#2A2A2A',  # Background color
    plot_bgcolor='#2A2A2A',   # Plot area color
    title='Recency vs Frequency by Cluster',
    title_font=dict(size=16, color='Teal'),  # Title color
    xaxis_title='Recency',
    xaxis_title_font=dict(size=14, color='Teal'),  # X-axis label color
    yaxis_title='Frequency',
    yaxis_title_font=dict(size=14, color='Teal'),  # Y-axis label color
    legend_title_font=dict(size=13, color='White'),  # Legend title color
    legend_font=dict(size=11, color='Plum'),  # Legend text color
    xaxis=dict(
        showgrid=True,
        zeroline=False,
        showline=False,
        linecolor='#FFFFFF',  # Outline color
        tickfont=dict(color='Plum'),  # Tick label color

    ),
    yaxis=dict(
        showgrid=True,
        zeroline=False,
        showline=False,
        linecolor='#FFFFFF',  # Outline color
        tickfont=dict(color='Plum')  # Tick label color
    ),
    margin=dict(l=40, r=40, t=40, b=40)  # Adjust margins
)

# Show the interactive plot
fig.show()